In [3]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# features.txt 파일에는 피처 이름 index와 피처명이 공백으로 분리되어 있음. 이를 DataFrame으로 로드.
feature_name_df = pd.read_csv('./human_activity/features.txt',sep='\s+',
                        header=None,names=['column_index','column_name'])

# 피처명 index를 제거하고, 피처명만 리스트 객체로 생성한 뒤 샘플로 10개만 추출
feature_name = feature_name_df.iloc[:, 1].values.tolist()
print('전체 피처명에서 10개만 추출:', feature_name[:10])

전체 피처명에서 10개만 추출: ['tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z', 'tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z', 'tBodyAcc-mad()-X', 'tBodyAcc-mad()-Y', 'tBodyAcc-mad()-Z', 'tBodyAcc-max()-X']


In [4]:
feature_dup_df = feature_name_df.groupby('column_name').count()
print(feature_dup_df[feature_dup_df['column_index'] > 1].count())
feature_dup_df[feature_dup_df['column_index'] > 1].head()

column_index    42
dtype: int64


,column_index
column_name,
"fBodyAcc-bandsEnergy()-1,16",3
"fBodyAcc-bandsEnergy()-1,24",3
"fBodyAcc-bandsEnergy()-1,8",3
"fBodyAcc-bandsEnergy()-17,24",3
"fBodyAcc-bandsEnergy()-17,32",3


In [5]:
def get_new_feature_name_df(old_feature_name_df):
    feature_dup_df = pd.DataFrame(data=old_feature_name_df.groupby('column_name').cumcount(),
                                  columns=['dup_cnt'])
    feature_dup_df = feature_dup_df.reset_index()
    new_feature_name_df = pd.merge(old_feature_name_df.reset_index(), feature_dup_df, how='outer')
    new_feature_name_df['column_name'] = new_feature_name_df[['column_name', 'dup_cnt']].apply(lambda x : x[0]+'_'+str(x[1]) 
                                                                                         if x[1] >0 else x[0] ,  axis=1)
    new_feature_name_df = new_feature_name_df.drop(['index'], axis=1)
    return new_feature_name_df

In [6]:
# cumcount()

df = pd.DataFrame([['a'],['b'],['c'],['a'],['a'],['b']],columns=['A'])
print(df)
df.groupby('A').cumcount()

   A
0  a
1  b
2  c
3  a
4  a
5  b


0    0
1    0
2    0
3    1
4    2
5    1
dtype: int64

In [7]:
import pandas as pd

def get_human_dataset( ):
    
    # 각 데이터 파일들은 공백으로 분리되어 있으므로 read_csv에서 공백 문자를 sep으로 할당.
    feature_name_df = pd.read_csv('./human_activity/features.txt',sep='\s+',
                        header=None,names=['column_index','column_name'])
    
    # 중복된 피처명을 수정하는 get_new_feature_name_df()를 이용, 신규 피처명 DataFrame생성. 
    new_feature_name_df = get_new_feature_name_df(feature_name_df)
    
    # DataFrame에 피처명을 컬럼으로 부여하기 위해 리스트 객체로 다시 변환
    feature_name = new_feature_name_df.iloc[:, 1].values.tolist()
    
    # 학습 피처 데이터 셋과 테스트 피처 데이터을 DataFrame으로 로딩. 컬럼명은 feature_name 적용
    X_train = pd.read_csv('./human_activity/train/X_train.txt',sep='\s+', names=feature_name )
    X_test = pd.read_csv('./human_activity/test/X_test.txt',sep='\s+', names=feature_name)
    
    # 학습 레이블과 테스트 레이블 데이터을 DataFrame으로 로딩하고 컬럼명은 action으로 부여
    y_train = pd.read_csv('./human_activity/train/y_train.txt',sep='\s+',header=None,names=['action'])
    y_test = pd.read_csv('./human_activity/test/y_test.txt',sep='\s+',header=None,names=['action'])
    
    # 로드된 학습/테스트용 DataFrame을 모두 반환 
    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = get_human_dataset()

C:\Users\rhals\AppData\Local\Temp\ipykernel_9384\4103130684.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if x[1] >0 else x[0] ,  axis=1)
C:\Users\rhals\AppData\Local\Temp\ipykernel_9384\4103130684.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  new_feature_name_df['column_name'] = new_feature_name_df[['column_name', 'dup_cnt']].apply(lambda x : x[0]+'_'+str(x[1])


In [8]:
print(X_train.info())
print(y_train.value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7352 entries, 0 to 7351
Columns: 561 entries, tBodyAcc-mean()-X to angle(Z,gravityMean)
dtypes: float64(561)
memory usage: 31.5 MB
None
action
6         1407
5         1374
4         1286
1         1226
2         1073
3          986
Name: count, dtype: int64


In [11]:
# decisiontreeclassifier로 동작 예측분류 수행하고 , 하이퍼 파라미터는 모두 디폴트 값, 하이퍼파라미터값추ㅜㄹ

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

dt_clf=DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train,y_train)
pred=dt_clf.predict(X_test)
score=accuracy_score(y_test,pred)

#정확도 값 출력
print(score)

# 기본파라미터
print(dt_clf.get_params())

0.8622327790973872
{'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'random_state': 42, 'splitter': 'best'}


In [12]:
from sklearn.model_selection import GridSearchCV

params={'max_depth':[6,8,10,12,16,24],'min_samples_split':[16]}

grid_cv=GridSearchCV(dt_clf,param_grid=params, scoring='accuracy',cv=5,verbose=1)
grid_cv.fit(X_train,y_train)

print(grid_cv.best_params_)
print(grid_cv.best_score_)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
{'max_depth': 10, 'min_samples_split': 16}
0.8506627450436326


In [14]:
cv_results_df =pd.DataFrame(grid_cv.cv_results_)
cv_results_df[['param_max_depth','mean_test_score']]

,param_max_depth,mean_test_score
0,8,0.849710
1,8,0.850932
2,10,0.850663
3,10,0.854335
4,12,0.841551
5,12,0.844679
6,16,0.847944
7,16,0.850663
8,24,0.847672
9,24,0.847262


In [32]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# min_samples_split을 16으로 고정하고 test데이터를 이용한 score값 추출하기

max_depths=[6,8,10,12,16,24]

for depths in max_depths:
    dt_clf=DecisionTreeClassifier(max_depth=depths,min_samples_split=16,random_state=42)
    dt_clf.fit(X_train,y_train)
    pred=dt_clf.predict(X_test)
    score=accuracy_score(y_test,pred)
    print(f'{depths} 정확도:{score}')

6 정확도:0.8547675602307431
8 정확도:0.8710553104852392
10 정확도:0.8676620291822192
12 정확도:0.8615541228367831
16 정확도:0.8618934509670851
24 정확도:0.8625721072276892


In [13]:
# max_depth, min_samples_split을 조정하기

params={'max_depth':[8,10,12,16,24],'min_samples_split':[16,24]}

grid_cv=GridSearchCV(dt_clf,param_grid=params,cv=5,scoring='accuracy',verbose=1)
grid_cv.fit(X_train,y_train)
print(grid_cv.best_score_)
print(grid_cv.best_params_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
0.8543348270647485
{'max_depth': 10, 'min_samples_split': 24}


In [15]:
best_grid_cv=grid_cv.best_estimator_
pred=best_grid_cv.predict(X_test)
score=accuracy_score(y_test,pred)
score

0.8676620291822192

In [40]:
# feature_importance 분기시에 특정 칼럼 데이터가 얼마나 영향을 미쳤는지?
# feature는 561개정도
best_grid_cv.feature_importances_

array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       1.71823547e-03, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 2.13804330e-03, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 9.78035435e-03, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 1.20545701e-03, 6.16972424e-03, 0.00000000e+00,
       2.48496852e-01, 6.51549652e-03, 6.11725280e-03, 1.63914560e-03,
      

In [1]:
import seaborn as sns

grid_importances_values = best_grid_cv.feature_importances_
#막대그래프로 표현하기 쉽도록 시리즈 정렬
grid_importances=pd.Series(grid_importances_values,index=X_train.columns)
# top20개만
grid_top_20=grid_importances.sort_values(ascending=False)[:20]


plt.figure(figsize=(8,6))
sns.barplot(x=grid_top_20,y=grid_top_20.index)
plt.title('feature_importance top20')
plt.show()

NameError: name 'best_grid_cv' is not defined